Copyright (c) 2025 Mitsuru Ohno  
Use of this source code is governed by a BSD-3-style  
license that can be found in the LICENSE file.  

## 当ノートブックのワークフロー  
速度定数を時間の関数として表現する場合の事例。  
1. 未知の速度定数を含む、csvに書き込んだ反応式を読み込む。  
2. 化学種の濃度の経時変化の実験データを読み込む。実験データには欠損を含んでもよい。  
2. RxnIVPsolv("sample_data.csv")でインスタンス化し、化学種ごとの微分型の反応速度式を作成する。速度定数が未知の素反応にはシンボリックな変数が割り振られる。  
3. 作成した微分方程式を、数値解析可能な形式にする  
4.  scipy.optimize.minimizeを使い、化学種ごとの濃度の経時変化が、実験値と近づくように、未知の速度定数を求める： get_ode_system()で必要なオブジェクトを取得。  
5. 経時変化を図示する： matplotlibで結果をプロット  

もしエラーが発生した場合は、debug_ode_system()メソッドで詳細な情報を確認できる。  


## 引用文献  
6) Kinetics of Polyesterification: A Study of the Effects of Molecular Weight and Viscosity on Reaction Rate
Paul J. Flory, Journal of the American Chemical Society 1939 61 (12), 3334-3340, DOI: 10.1021/ja01267a030  
https://doi.org/10.1021/ja01267a030  

### 事前のデータ処理  
Table Iのデータに以下の処理を施した。内容はref6.xlsxを参照のこと。  
- diethylene glycol (DEG: MW 106.12), adipic acid (AdA: MW 146.14)とした。CRUのMWは 106.12 + 146.12 - 18.02 x 2 = 216.22とした。  
- 初期濃度t = 0は、モノマーであるDEGとAdAが、1,000 g 中に等物質量ずつ含まれているとした。ともに3.964 molずつで、 DEG (420.7 g)、AdA (579.3 g) になる。  
- polyesterのDPnからMnを算出した。DPn=0の場合を除き、Mn = DPn x 216.22 + 18.02 とした。  
- 触媒の重量は無視できるとした。  
- 各反応時間でのモノマー残量は表中のρ（転嫁率に相当）から算出した。  
- 発生した水は速やかに系から除去されるとし、その重量をρから算出した。t=0での総重量 1,000 gから、発生した水の重量を減じ、各時間での総重量を求めた。  
- DEG, AdA以外はすべてpolyesterと考えてその重量を算出し、各時間でのMnで除して主鎖の物質量とした。  
- 各時間での、各化学種の物質量を、その時点での総重量/1000 で除し、濃度 \[mol/1000g\] とした。  

計算の詳細は './sample_data/ref6/sample_rxn_ref6.xlsx' を参照。


## <span style="color: red; font-weight: bold;">under preparaion</span>
### 考え方  
解析はMnの成長速度を見込むことを目標にした。ただし、数値のオーダーを、化学種の濃度のそれと揃えるため、Mnを1000で除した値（mMn: ミリMn）を用いた。  

**主鎖成長の速度定数**  
主鎖の成長の速度定数kpは、モノマー同士の速度定数kmに対する時間の関数として表現でき、ある時間でのkpは  

$kp = {a}{k_m}{exp({-b}{t})}$ (eq 1)  

で表されると仮定した。  
重合初期では反応速度はモノマー濃度に支配され、重合の進行に伴う粘度上昇や分子量増大により拡散係数および実効的な頻度因子が低下する。この効果を、反応速度定数 k の時間依存的変化として表現する考え方である。  
よって、フィッティングでは、kmとaを求めることとなる。  
この際、モノマー同士の反応により "olg" （オリゴマー）が生成し、olg同志の反応によりポリマーが生成する、と素反応を設定した。すなわち、モノマーが主鎖末端に縮合して主鎖成長するパスは無視しており、かつオリゴマーの反応性は鎖長によらず同一と暗に仮定していることになる。なお、olg濃度はt=0で0、他は全て未知（空欄）とした。  

**目的変数**  
目的変数は、数平均分子量そのものではなく、数平均分子量の逆数とした。実際には、前述のとおり、数値のオーダーを、化学種の濃度のそれと揃えるため、Mnを1000で除した値（recip_mMn）を用いた。  
Carothers equationと同様の考え方により、本系での数平均分子量Mnは、

$DP_n = \frac{1}{(1 - p)^2} = \frac{M_n}{M_{\mathrm{CRU}}}$ （eq 2: ただし $M_{\mathrm{CRU}}$ は CRU の分子量）  

となる。また、形式上的には、仕込みのCRUの濃度は、仕込みAdAのそれと等しいと考えることができるため、  

$DP_n = \frac{[AdA]_0}{[polym]}$ (eq 3)  

である。eq 2 および eq 3 から、  

$ \frac{M_n}{M_{\mathrm{CRU}}} = \frac{[AdA]_0}{[polym]}$ (eq 4)  

よって、  

$ [polym] = \frac{M_{\mathrm{CRU}}[AdA]_0}{M_n}$ (eq 5)  

となる。ここで、 $M_{\mathrm{CRU}}$ 、 $[AdA]_0$ は実験条件から求めることができる。従って、理論的的には、ポリマー濃度が求まれば、数平均分子量が求まるはずである。  
ただし、現行バージョンのrxnfitでは、濃度項のべき乗は取り扱えないため、実験データ側で逆数として入力した。また、eq 5 の $\frac{M_{\mathrm{CRU}}}{M_n}$ は定数であり、 eq 1の a に含んで考慮できると考えた。  


## 反応式, 速度定数の定義を記載したcsvファイルを指定する  

In [ ]:
file_path_rxn = './sample_data/ref6/sample_rxn_ref6.csv'  # CSVファイルのパスを指定
file_path_k = './sample_data/ref6/sample_k_ref6.csv'

## 反応速度式をscipy.integrate.solve_ivpで処理できる連立微分方程式にする  

In [ ]:
import time

import numpy as np
import pandas as pd

from rxnfit import RxnODEbuild, SolverConfig, RxnODEsolver
from rxnfit.expdata_reader import expdata_read
from rxnfit.expdata_fit_sci import ExpDataFitSci
from rxnfit.p0_opt_fit import P0OptFit

# 反応速度式の作成

In [ ]:
builded_rxnode = RxnODEbuild(
    file_path_rxn, 
    rate_const_overrides=file_path_k) # 時間の関数で表現される速度定数の定義場所を指定  

In [ ]:
builded_rxnode.get_ode_info(debug_info=True)

In [ ]:
# 作成した微分方程式
builded_rxnode.get_ode_system()[0]

In [ ]:
# 速度定数の確認
print(builded_rxnode.rate_consts_dict)

check_type = [v for v in builded_rxnode.rate_consts_dict.values()]
[type(e) for e in check_type]

## 経時変化の実験データを読み込み　　
### データフレーム化  

In [ ]:
file_path_data = './sample_data/ref6/sample_timecourse_ref6.csv' # データファイルのパス
df1 = pd.read_csv(file_path_data)

expdata_read([df1,])

### フィッティング  
ExpDataFitSci でシンボリックな速度定数をフィッティング。  


In [ ]:
t_cpu_start = time.process_time() # CPU時間計測

# P0OptFit: RxnODEbuild インスタンスと df_list を渡す（他は最小限）
opt = P0OptFit(builded_rxnode, [df1,], seed=123)
result_dict, fit_metrics = opt.optimize(n_trials=5)

# プロット用に最適 p0 で ExpDataFitSci を1回実行し、config を取得
keys = builded_rxnode.get_symbolic_rate_const_keys()
p0_best = [result_dict[k][1] for k in keys]
t_range = (0, float(df1.iloc[:, 0].max()))
fit_sci = ExpDataFitSci(builded_rxnode, [df1, ], t_range)
fit_sci.run_fit(p0_best, use_log_fit=True, lower_bound=1e-10)

t_cpu_end = time.process_time()
elapsed = t_cpu_end - t_cpu_start
h, r = divmod(int(elapsed), 3600)
m, s = divmod(r, 60)
print(f"CPU time: {h:02d}:{m:02d}:{s:02d}")

In [ ]:
# 経時変化を図示（plot_fitted_solution を使用）
fit_sci.plot_fitted_solution(expdata_df=df1, species=['AdA', 'olg', 'recip_mMn'])

### 初期値最適化のログの確認  

In [ ]:
logs = opt.optuna_log()

rows = []
for d in logs:
    row = {"trial_No": d["trial_No"], "rss": d["rss"], "state": d["state"]}
    row.update(d["params"])
    rows.append(row)
df_log = pd.DataFrame(rows)
df_log